## Конфигурация

In [5]:
import asyncio
import json
from pathlib import Path
from tqdm import tqdm
from typing import Tuple, Set, Dict

import polars as pl
from openai import AsyncOpenAI

CATEGORY = "Pet_Supplies"
DATA_DIR = Path.cwd()
CHECKPOINT_PATH = "checkpoint/clean_description_and_title.jsonl"
OUTPUT_PATH = f"prepared/{CATEGORY}_items_cleaned.parquet"
INPUT_PATH = f"prepared/{CATEGORY}_items.parquet"

MAX_TITLE_LEN = 200
MAX_DESC_LEN = 2000
MAX_FEATURES_LEN = 2000

OPENROUTER_API_KEY = __import__('os').environ['OPENROUTER_API_KEY']
MODEL = "google/gemini-2.5-flash-lite"
CONCURRENCY = 50

client = AsyncOpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
)

## Промпт

In [6]:
SYSTEM_PROMPT = """You normalize e-commerce product metadata. Not a copywriter.

Return STRICT JSON:
{"clean_title": string, "clean_description": string, "clean_features": [string]}

Rules:
- Use ONLY facts present in the input. Do NOT add, infer, generalize, or invent anything.
- Preserve all numbers, units, sizes, materials, limits, warranty, country of origin, and brand names.
- Fix grammar, spelling, punctuation.
- Remove HTML, entities, emojis/checkmarks, duplicate words/phrases, formatting noise, ALL CAPS shouting.
- If text ends with "...", complete ONLY from existing context.

Title:
- Remove promo noise (NEW, SALE, LIMITED, etc.).
- Remove seller/store/platform tags.
- Use Title Case.
- Keep brand, model, key specs.

Description:
- Keep all factual info.
- Under 200 words.
- Convert lists/bullets into fluent sentences.

Features:
- Output 3–12 concise bullets when possible.
- Each bullet should be factual and short (3–18 words if possible).
- Remove non-factual marketing fluff (e.g., “best”, “perfect”, “leader”, “focus of the scene”, “after-sales service”)
  unless it contains a concrete verifiable fact (e.g., “since 1979”, “90-day guarantee”).
- Deduplicate near-identical bullets.
- If features are empty or missing, return [].

Return ONLY JSON.
"""

## Функции для обработки данных

In [7]:
def load_checkpoint_index(
    path: Path,
    id_key: str = "parent_asin",
) -> Tuple[Set[str], Dict[str, dict]]:
    """
    Один проход по JSONL чекпоинту:
    - processed_success: asin, которые успешно обработаны (error == None)
    - latest_by_asin: последняя запись по asin (для merge при рестарте)
    """
    processed_success: Set[str] = set()
    latest_by_asin: Dict[str, dict] = {}

    if not path.exists():
        return processed_success, latest_by_asin

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except Exception:
                continue

            asin = obj.get(id_key)
            if not asin:
                continue

            latest_by_asin[asin] = obj
            if obj.get("error") is None:
                processed_success.add(asin)

    return processed_success, latest_by_asin


def append_jsonl(path: Path, obj: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")


def strip_code_fences(s: str) -> str:
    s = (s or "").strip()
    if s.startswith("```"):
        s = s.split("\n", 1)[1]
        if s.endswith("```"):
            s = s[:-3]
    return s.strip()


async def clean_one(parent_asin: str, title: str, description: str, features_text: str) -> dict:
    try:
        title = (title or "").strip()
        description = (description or "").strip()
        features_text = (features_text or "").strip()

        if len(title) > MAX_TITLE_LEN:
            title = title[:MAX_TITLE_LEN]
        if len(description) > MAX_DESC_LEN:
            description = description[:MAX_DESC_LEN]
        if len(features_text) > MAX_FEATURES_LEN:
            features_text = features_text[:MAX_FEATURES_LEN]

        user_prompt = f"""TITLE:
{title}

DESCRIPTION:
{description}

FEATURES_TEXT:
{features_text}
"""
        resp = await client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0,
        )

        content = strip_code_fences(resp.choices[0].message.content or "")
        data = json.loads(content)

        feats = data.get("clean_features")
        if not isinstance(feats, list):
            feats = []
        feats = [str(x).strip() for x in feats if str(x).strip()]

        return {
            "parent_asin": parent_asin,
            "clean_title": (data.get("clean_title") or "").strip() or None,
            "clean_description": (data.get("clean_description") or "").strip() or None,
            "clean_features": feats,
            "error": None,
        }

    except Exception as e:
        return {
            "parent_asin": parent_asin,
            "clean_title": None,
            "clean_description": None,
            "clean_features": None,
            "error": f"{type(e).__name__}: {e}",
        }


async def clean_polars_df(
    df: pl.DataFrame,
    checkpoint_path: str | Path,
    id_col: str = "parent_asin",
    title_col: str = "title",
    desc_col: str = "description_text",
    features_col: str = "features_text",
    concurrency: int = 8,
) -> pl.DataFrame:
    concurrency = max(1, int(concurrency))
    checkpoint = Path(checkpoint_path)

    processed, latest_map = load_checkpoint_index(checkpoint, id_key=id_col)

    todo = df.filter(~pl.col(id_col).is_in(list(processed)))
    total = todo.height

    async def run_row(row: dict) -> dict:
        return await clean_one(
            row[id_col],
            row.get(title_col),
            row.get(desc_col),
            row.get(features_col),
        )

    new_results: list[dict] = []
    pending: list[asyncio.Task] = []

    with tqdm(total=total, desc="Cleaning items") as pbar:
        for row in todo.select([id_col, title_col, desc_col, features_col]).iter_rows(named=True):
            pending.append(asyncio.create_task(run_row(row)))

            if len(pending) >= concurrency:
                done = await asyncio.gather(*pending)
                for r in done:
                    append_jsonl(checkpoint, r)
                    new_results.append(r)
                pbar.update(len(done))
                pending.clear()

        if pending:
            done = await asyncio.gather(*pending)
            for r in done:
                append_jsonl(checkpoint, r)
                new_results.append(r)
            pbar.update(len(done))

    for r in new_results:
        asin = r.get(id_col)
        if asin:
            latest_map[asin] = r

    if latest_map:
        all_df = pl.DataFrame(list(latest_map.values()))

        for c in [id_col, "clean_title", "clean_description", "clean_features"]:
            if c not in all_df.columns:
                all_df = all_df.with_columns(pl.lit(None).alias(c))

        all_df = all_df.select([id_col, "clean_title", "clean_description", "clean_features"])

        return df.join(all_df, on=id_col, how="left")

    return df.with_columns([
        pl.lit(None).alias("clean_title"),
        pl.lit(None).alias("clean_description"),
        pl.lit(None).alias("clean_features"),
    ])

In [8]:
items_df = pl.read_parquet(INPUT_PATH)

In [9]:
items_df

parent_asin,main_category,title,average_rating,rating_number,features_text,description_text,price,store,categories_text,item_context
str,str,str,f32,i32,str,str,f32,str,str,str
"""B00XJG2SLG""","""Pet Supplies""","""Hurtta Pet Collection 14-Inch …",4.4,166,"""Made from highly durable Neopr…","""Hurtta harnesses are suitable …",24.950001,"""Hurtta""","""Pet Supplies > Dogs > Collars,…","""Product: Hurtta Pet Collection…"
"""B01MQTWB5H""","""Pet Supplies""","""4 Pack - 4 Inch Ring Filter So…",4.4,84,"""Micron filter bags provide exc…","""Micron filter bags provide exc…",15.0,"""Encompass All""","""""","""Product: 4 Pack - 4 Inch Ring …"
"""B00F3JRLYQ""","""Pet Supplies""","""Wysong Optimal Adult Canine Fo…",4.4,75,"""42% Protein With High Levels O…","""For the past 35 years the Wyso…",17.459999,"""Wysong""","""Pet Supplies > Dogs > Food > D…","""Product: Wysong Optimal Adult …"
"""B09J5FJYMJ""","""Pet Supplies""","""Nutramax Dasuquin Joint Health…",4.3,641,"""Joint Health Support for Cats:…","""Dasuquin for Cats Soft Chews i…",9.68,"""Dasuquin""","""Pet Supplies > Cats > Health S…","""Product: Nutramax Dasuquin Joi…"
"""B08WZ1Z9ZF""","""Pet Supplies""","""Batiyeer 4 Strings in 4 Pieces…",4.0,95,"""Reliable to use: due to the qu…","""Features: Remind other animal…",null,"""Batiyeer""","""Pet Supplies > Dogs > Collars,…","""Product: Batiyeer 4 Strings in…"
…,…,…,…,…,…,…,…,…,…,…
"""B0013ZEGWO""","""Pet Supplies""","""Sera 7112 Koi Royal Mini 1.8 l…",4.6,1054,"""Staple food for Koi Strengthen…","""Sera Koi Royal is the staple f…",20.75,"""Sera""","""Pet Supplies > Fish & Aquatic …","""Product: Sera 7112 Koi Royal M…"
"""B099J628C8""","""""","""Guess What? My Mom is Pregnant…",4.6,161,"""Texture of material：We use 100…","""Trendy Designs: Our goal was t…",10.99,"""Yangmics Direct""","""Pet Supplies > Dogs > Apparel …","""Product: Guess What? My Mom is…"
"""B0BJFS2B7J""","""Pet Supplies""","""DIELLY 5 Set AI Artificial Ins…",4.5,27,"""【Dog AI Artificial Inseminatio…","""5 Set AI Artificial Inseminati…",9.88,"""DIELLY""","""Pet Supplies > Dogs > Training…","""Product: DIELLY 5 Set AI Artif…"


In [10]:
cleaned = await clean_polars_df(items_df, CHECKPOINT_PATH, concurrency=CONCURRENCY)

Cleaning items: 100%|██████████| 25398/25398 [28:32<00:00, 14.83it/s]


In [11]:
cleaned.write_parquet(OUTPUT_PATH)

In [13]:
cleaned

parent_asin,main_category,title,average_rating,rating_number,features_text,description_text,price,store,categories_text,item_context,clean_title,clean_description,clean_features
str,str,str,f32,i32,str,str,f32,str,str,str,str,str,list[str]
"""B00XJG2SLG""","""Pet Supplies""","""Hurtta Pet Collection 14-Inch …",4.4,166,"""Made from highly durable Neopr…","""Hurtta harnesses are suitable …",24.950001,"""Hurtta""","""Pet Supplies > Dogs > Collars,…","""Product: Hurtta Pet Collection…","""Hurtta Pet Collection 14-Inch …","""Hurtta harnesses are suitable …","[""Made from durable Neoprene"", ""Fitted with efficient 3M reflectors"", … ""Protection from the clip buckle""]"
"""B01MQTWB5H""","""Pet Supplies""","""4 Pack - 4 Inch Ring Filter So…",4.4,84,"""Micron filter bags provide exc…","""Micron filter bags provide exc…",15.0,"""Encompass All""","""""","""Product: 4 Pack - 4 Inch Ring …","""4 Pack - 4 Inch Ring Filter So…","""These 4-inch ring filter socks…","[""4 pack of 4-inch ring filter socks"", ""200-micron filtration"", … ""Durable and reusable""]"
"""B00F3JRLYQ""","""Pet Supplies""","""Wysong Optimal Adult Canine Fo…",4.4,75,"""42% Protein With High Levels O…","""For the past 35 years the Wyso…",17.459999,"""Wysong""","""Pet Supplies > Dogs > Food > D…","""Product: Wysong Optimal Adult …","""Wysong Optimal Adult Canine Fo…","""Wysong Optimal Adult super pre…","[""42% protein with high levels of fresh/frozen and dried meats"", ""Natural protein and fat"", … ""Wysong has been a leader in pet nutrition since 1979""]"
"""B09J5FJYMJ""","""Pet Supplies""","""Nutramax Dasuquin Joint Health…",4.3,641,"""Joint Health Support for Cats:…","""Dasuquin for Cats Soft Chews i…",9.68,"""Dasuquin""","""Pet Supplies > Cats > Health S…","""Product: Nutramax Dasuquin Joi…","""Nutramax Dasuquin Joint Health…","""Dasuquin for Cats Soft Chews i…","[""Joint health support for cats"", ""Contains ASU, glucosamine, and chondroitin sulfate"", … ""Veterinarian formulated with high-quality ingredients""]"
"""B08WZ1Z9ZF""","""Pet Supplies""","""Batiyeer 4 Strings in 4 Pieces…",4.0,95,"""Reliable to use: due to the qu…","""Features: Remind other animal…",null,"""Batiyeer""","""Pet Supplies > Dogs > Collars,…","""Product: Batiyeer 4 Strings in…","""Batiyeer 4 Strings in 4 Pieces…","""These collar bells are made of…","[""Made of quality brass material."", ""Durable, tough, and not easy to break or corrode."", … ""Enough for daily use and replacement.""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""B0013ZEGWO""","""Pet Supplies""","""Sera 7112 Koi Royal Mini 1.8 l…",4.6,1054,"""Staple food for Koi Strengthen…","""Sera Koi Royal is the staple f…",20.75,"""Sera""","""Pet Supplies > Fish & Aquatic …","""Product: Sera 7112 Koi Royal M…","""Sera 7112 Koi Royal Mini 1.8 l…","""Sera Koi Royal is a staple foo…","[""Staple food for Koi"", ""Strengthens the immune system"", … ""For Koi up to 5 inches""]"
"""B099J628C8""","""""","""Guess What? My Mom is Pregnant…",4.6,161,"""Texture of material：We use 100…","""Trendy Designs: Our goal was t…",10.99,"""Yangmics Direct""","""Pet Supplies > Dogs > Apparel …","""Product: Guess What? My Mom is…","""Pregnancy Announcement Dog Ban…","""These trendy, neutral bandanas…","[""Made from 100% soft spun polyester"", ""Breathable material keeps dogs cool"", … ""Pre-folded and easy to wear""]"
"""B0BJFS2B7J""","""Pet Supplies""","""DIELLY 5 Set AI Artificial Ins…",4.5,27,"""【Dog AI Artificial Inseminatio…","""5 Set AI Artificial Inseminati…",9.88,"""DIELLY""","""Pet Supplies > Dogs > Training…","""Product: DIELLY 5 Set AI Artif…","""DieLLY 5 Set AI Artificial Ins…","""This dog artificial inseminati…","[""Includes 5 AI rods tubes, 5 centrifuge tubes, 5 semen collection bags, 5 (5ml) syringes, 5 gloves."", ""Dog fertilization catheter is durable, smooth, and bendable."", … ""Syringes are individually sealed and made of durable plastic.""]"
